In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from langchain_chroma import Chroma
from typing import List
from dotenv import load_dotenv
import os
import json
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title



load_dotenv()

c:\Users\34656\OneDrive\Escritorio\Research\TFM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Partición del documento

In [2]:
def partition_document(file_path: str):
    """
    Carga el documento y extrae sus elementos usando una estrategia de alta resolución
    para capturar tablas e imágenes en base64.
    """
    if file_path.endswith('.pdf'):
        elements = partition_pdf(
            filename=file_path,
            strategy="hi_res",
            infer_table_structure=True,
            extract_image_block_types=["Image"],
            extract_image_block_to_payload=True
        )
        return elements

In [10]:
file_path = "C:/Users/34656/OneDrive/Escritorio/Research/TFM/RAG/data/docs/attention.pdf"
elements = partition_document(file_path)

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 1567.76it/s]


In [ ]:
elements

## 2. Creando chunks a partir de elementos unstructured

In [16]:
def create_chunks_by_title(elementos): 
    chunks = chunk_by_title(elementos, max_characters=3000, new_after_n_chars=2400, combine_text_under_n_chars=500)
    return chunks

chunks = create_chunks_by_title(elements)
chunks 

## 3. Separación del contenido 

In [ ]:
def separate_content_types(chunk): 
    """Analiza qué tipo de contenido hay en cada chunk"""
    content_data = {
        'text': chunk.text, 
        'tables': [], 
        'images': [],
        'types': ['text']
    }
    
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'): 
        for element in chunk.metadata.orig_elements: 
            element_type = type(element).__name__
            
            # Handle tables 
            if element_type == 'Table': 
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
                
            # Handle images
            elif element_type == 'Image': 
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
                    
    content_data['types'] = list(set(content_data['types']))
    return content_data

## 4. Creación de un resumen hecho con IA

In [ ]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str: 
    """Crea un resumen mejorado con IA para contenido mixto"""
    try: 
        
        # Cargar el LLM y la api key
        LLM = os.getenv("LLM")
        api_key = os.getenv("GOOGLE_API_KEY")
        
        # Initialize the LLM with Google
        llm = ChatGoogleGenerativeAI(model=LLM, api_key = api_key)
        
        # Build the text prompt
        prompt_text = f"""Estás creando una descripción que permite realizar búsquedas para recuperar el contenido del documento.
        
        CONTENIDO PARA ANALIZAR: 
        CONTENIDO TEXTUAL: 
        {text}
        """
        
        if tables: 
            prompt_text += "TABLAS:\n"
            for i, table in enumerate(tables): 
                prompt_text += f"Tabla {i+1}:\n{table}\n\n"
                
                prompt_text += """
                TU TAREA: 
                Genera una descripción completa y consultable que incluya: 

                1. Datos clave, cifras y puntos de datos del texto y las tablas.
                2. Temas y conceptos principales tratados.
                3. Preguntas que este contenido podría responder. 
                4. Análisis visual del contenido (gráficos, diagramas, patrones en imágenes).
        
                Hazla detallada y consultable; prioriza la claridad y descripción sobre la brevedad.
                
                DESCRIPCIÓN CONSULTABLE:"""
                
        message_content = [{"type": "text", "text": prompt_text}]
        
        for image_base64 in images: 
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
    
    except Exception as e: 
        print(f"""Error generando descripción: {e}""")
        
        summary = f"{text[:300]}..."
        if tables: 
            summary += f" [Contiene {len(tables)} tabla(s)]"
        if images: 
            summary += f" [Contiene {len(images)} imagen(es)]"
        return summary

## 5. Añadir resumenes con contenido del chunk

In [ ]:
def summarise_chunks(chunks): 
        """Procesa todos los chunks con resúmenes IA"""
        print("Procesando chunks con resúmenes IA... ")
    
        langchain_documents = []
        total_chunks = len(chunks)
    
        for i, chunk in enumerate(chunks):
            current_chunk = i + 1
            print(f" Procesando chunk {current_chunk} / {total_chunks}")
            
            # Analyze chunk content
            content_data = separate_content_types(chunk)
            
            # Debug prints
            print(f"    Tipos encontrados: {content_data['types']}")
            print(f"    Tablas: {len(content_data['tables'])}, Imágenes: {len(content_data['images'])}")
            
            # Create AI-Enhanced summary if chunk has tables/images
            if content_data['tables'] or content_data['images']: 
                print(f" Creando resumen con IA para contenido mixto...")
                
                try: 
                    enhanced_content = create_ai_enhanced_summary(
                        content_data['text'],
                        content_data['tables'],
                        content_data['images']
                    )    
                    
                    print(f"    Resumen IA creado exitosamente")
                    print(f"    Contenido mejorado: {enhanced_content[:200]}")
                
                except Exception as e: 
                    print(f"     Resumen IA falló: {e}")
                    enhanced_content = content_data['text']
            
            else: 
                print(f"    Usando texto crudo (sin imágenes o tablas)")
                enhanced_content = content_data['text']
                    
            # Forzamos la conversión a string por si viene algún otro tipo de objeto
            enhanced_content = str(enhanced_content).strip()
            # -----------------------------------------------

            if enhanced_content: 
                
                doc = Document(
                    page_content=enhanced_content, 
                    metadata={
                        "original_content": json.dumps({
                            "raw_text": content_data['text'],
                            "tables_html": content_data['tables'],
                            "images_base64": content_data['images']  
                        })
                    }
                )
                langchain_documents.append(doc)
            
            else: 
                print(f"    [Skipped] El chunk {current_chunk} está vacío tras el procesado.")
            
            
        print(f"Processed {len(langchain_documents)} chunks")
        return langchain_documents

In [21]:
processed_chunks = summarise_chunks(chunks)

Procesando chunks con resúmenes IA... 
 Procesando chunk 1 / 25
    Tipos encontrados: ['text']
    Tablas: 0, Imágenes: 0
    Using raw text (no images or tables)
 Procesando chunk 2 / 25
    Tipos encontrados: ['text']
    Tablas: 0, Imágenes: 0
    Using raw text (no images or tables)
 Procesando chunk 3 / 25
    Tipos encontrados: ['text']
    Tablas: 0, Imágenes: 0
    Using raw text (no images or tables)
 Procesando chunk 4 / 25
    Tipos encontrados: ['text']
    Tablas: 0, Imágenes: 0
    Using raw text (no images or tables)
 Procesando chunk 5 / 25
    Tipos encontrados: ['text', 'image']
    Tablas: 0, Imágenes: 1
 Creando resumen con IA para contenido mixto...
    Resumen IA creado exitosamente
    Contenido mejorado: [{'type': 'text', 'text': 'Esta es una descripción técnica optimizada para motores de búsqueda y sistemas de recuperación de información, basada en el contenido proporcionado:\n\n---\n\n### **Título: Arquitectura del Modelo Transformer - Estructura Encoder-Deco

## 6. Exportar chunks a formato JSON

In [22]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Exporta una lista de chunks a un archivo JSON."""
    
    export_data = []
    for i, doc in enumerate(chunks): 
        chunk_data = {
            "chunk_id": i + 1, 
            "enhanced_content": doc.page_content, 
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
        
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f: 
        json.dump(export_data, f, ensure_ascii=False, indent=2)
    
    print(f"Exported {len(export_data)} chunks to {filename}")
    return export_data

        
json_data = export_chunks_to_json(processed_chunks) 

Exported 25 chunks to chunks_export.json


## 7. Crear Vector Store

In [25]:
def create_vector_store(documents, persist_directory = "data/chromadb"): 
    """Creates a vector store from the given documents."""
    print("Creating embeddings and storing in ChromaDB...")
    
    api_key = os.getenv("GOOGLE_API_KEY")
    embedding_model = GoogleGenerativeAIEmbeddings(api_key=api_key, model="gemini-embedding-2-preview")
    
    # Create ChromaDB vector store
    print("--Creando vector store--")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    
    print("--Vector Store creada--")
    
    return vectorstore

In [26]:
db = create_vector_store(processed_chunks)

# After the retrieval
query = "Explicame la capa de encoder"
retriever = db.as_retriever(search_kargs = {"k":3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

Creating embeddings and storing in ChromaDB...
--Creando vector store--


IndexError: list index out of range

## 8. Correr pipeline entera

In [ ]:
def run_complete_ingestion_pipline(pdf_path:str): 
    """Run the complete RAG ingestion pipeline."""
    print("Starting RAG ingestion pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chromadb")
    
    print("Pipeline completed successfully")
    return db

## 9. Generar respuesta final

In [ ]:
def generate_final_answer(chunks, query): 
    """Generate final answer using multimodal content"""
    
    try: 
        llm = ChatOllama(model = "gpt-oss:20b", base_url= "")
        
        prompt = f""""Based on the following documents, please answer this question: {query}
        
        CONTENT TO ANALYZE: 
        """
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"---Document {i + 1} ---\n"
            
            if "original_content" in chunk.metadata: 
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text: 
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                    
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html: 
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate (tables_html): 
                        prompt_text += f"Table {j + 1}:\n{table}\n\n"
                        
            prompt_text += "\n"
            
        prompt_text += """ 
Please provide a clear, comprehensive answer using the text, tables and images above it. If the documents don't contain any relevant information, please respond with "No relevant information found."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks: 
            if "original_content" in chunk.metadata: 
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64: 
                    message_content.append({
                        "type": "image_url", 
                        "image_url": {"url": f"data:image/jpeg;base64, {image_base64}"}
                    })
                    
        # Send to AI and get response 
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
    
    except Exception as e: 
        print(f"Answer generation failed: {e}")     
        return "Sorry, I encountered an error while generating the answer"

In [ ]:
# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)